# **Testing Model with Preprocessing**

In [1]:
import os
import cv2
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from PIL import Image, ImageEnhance
from sklearn.metrics import accuracy_score

from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.applications.vgg16 import preprocess_input

## **Image Preprocessing Functions**

In [2]:
def enhance_image(image):
    img = Image.fromarray(image)
    
    brightness = ImageEnhance.Brightness(img)
    img_enhanced = brightness.enhance(1.1)
    
    saturation = ImageEnhance.Color(img_enhanced)
    img_enhanced = saturation.enhance(1.1)
    
    contrast = ImageEnhance.Contrast(img_enhanced)
    img_enhanced = contrast.enhance(1.1)
    
    return np.array(img_enhanced)

In [3]:
def sharpen_image(image):
    laplacian = cv2.Laplacian(image, cv2.CV_64F)
    laplacian_norm = cv2.normalize(laplacian, None, 0, 255, cv2.NORM_MINMAX)
    
    sharpened = cv2.addWeighted(image, 0.8, -laplacian_norm.astype(np.uint8), 0.2, 0)
    img_sharpened = np.clip(sharpened, 0, 255).astype(np.uint8)
    
    return img_sharpened

## **Initialising Variables and Functions**

In [4]:
LABELS = ['FreshApple', 'FreshBanana', 'FreshGrape', 'FreshGuava', 'FreshJujube', 'FreshOrange', 'FreshPomegranate', 'FreshStrawberry',
          'RottenApple', 'RottenBanana', 'RottenGrape', 'RottenGuava', 'RottenJujube', 'RottenOrange', 'RottenPomegranate', 'RottenStrawberry']
INPUT_SIZE = (256, 256)

In [5]:
def load_dataset(dataset_path, input_size):
    images = []
    labels = []

    for img_file in os.listdir(dataset_path):
        img_path = os.path.join(dataset_path, img_file)
        img = cv2.imread(img_path)
        
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, input_size)
            img_processed = enhance_image(img)
            img_processed = sharpen_image(img_processed)
            
            images.append(img_processed)
            labels.append('FreshStrawberry')
        else:
            print(f"Warning: Could not read image {img_path}")
    
    return np.array(images), np.array(labels)

In [6]:
def predict(model):
    y_pred_probs = model.predict(X_test)
    y_pred_idx = np.argmax(y_pred_probs, axis=1)
    y_pred = [LABELS[i] for i in y_pred_idx]

    return y_pred

In [7]:
def print_accuracy(y_test, y_pred):
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {(accuracy * 100):.2f}%")

## **Prediction on Self-Collected Dataset**

In [8]:
DATASET_PATH = "Self-Collected Dataset"
X_test, y_test = load_dataset(DATASET_PATH, INPUT_SIZE)
X_test = X_test.astype('float32') / 255.0

print("Shape of X test :", X_test.shape, '\n')
print("Shape of y test :", y_test.shape)

Shape of X test : (200, 256, 256, 3) 

Shape of y test : (200,)


### **CNN**

In [9]:
MODEL_PATH = "Models WithPreprocessing/CNN_WithPreprocessing.h5"
model = load_model(MODEL_PATH)

y_pred = predict(model)
print_accuracy(y_test, y_pred)

7/7 ━━━━━━━━━━━━━━━━━━━━ 143s 19s/step
Accuracy: 18.00%


### **ResNet50**

In [10]:
MODEL_PATH = "Models WithPreprocessing/ResNet50_WithPreprocessing.h5"
model = load_model(MODEL_PATH, custom_objects={'preprocess_input': preprocess_input})

y_pred = predict(model)
print_accuracy(y_test, y_pred)

7/7 ━━━━━━━━━━━━━━━━━━━━ 59s 8s/step
Accuracy: 53.00%


### **VGG16**

In [11]:
MODEL_PATH = "Models WithPreprocessing/VGG16_WithPreprocessing.h5"
model = load_model(MODEL_PATH, custom_objects={'preprocess_input': preprocess_input})

y_pred = predict(model)
print_accuracy(y_test, y_pred)

7/7 ━━━━━━━━━━━━━━━━━━━━ 270s 35s/step
Accuracy: 47.50%


## **Prediction on 1D_Group6 Dataset**

In [12]:
DATASET_PATH = "1D_Group6"
X_test, y_test = load_dataset(DATASET_PATH, INPUT_SIZE)
X_test = X_test.astype('float32') / 255.0

print("Shape of X test :", X_test.shape, '\n')
print("Shape of y test :", y_test.shape)

Shape of X test : (206, 256, 256, 3) 

Shape of y test : (206,)


### **CNN**

In [13]:
MODEL_PATH = "Models WithPreprocessing/CNN_WithPreprocessing.h5"
model = load_model(MODEL_PATH)

y_pred = predict(model)
print_accuracy(y_test, y_pred)

7/7 ━━━━━━━━━━━━━━━━━━━━ 151s 21s/step
Accuracy: 12.62%


### **ResNet50**

In [14]:
MODEL_PATH = "Models WithPreprocessing/ResNet50_WithPreprocessing.h5"
model = load_model(MODEL_PATH, custom_objects={'preprocess_input': preprocess_input})

y_pred = predict(model)
print_accuracy(y_test, y_pred)

7/7 ━━━━━━━━━━━━━━━━━━━━ 107s 13s/step
Accuracy: 55.83%


### **VGG16**

In [15]:
MODEL_PATH = "Models WithPreprocessing/VGG16_WithPreprocessing.h5"
model = load_model(MODEL_PATH, custom_objects={'preprocess_input': preprocess_input})

y_pred = predict(model)
print_accuracy(y_test, y_pred)

7/7 ━━━━━━━━━━━━━━━━━━━━ 333s 44s/step
Accuracy: 58.25%
